# 🚗 Car Price Prediction — Machine Learning Project
> **Dataset**: [Kaggle — vijayaadithyanvg/car-price-predictionused-cars](https://www.kaggle.com/datasets/vijayaadithyanvg/car-price-predictionused-cars)
> 
> **Libraries**: Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn

---
### 📌 Objectives
1. Explore and visualise the used car dataset
2. Engineer meaningful features (Brand Goodwill, Car Age, Power/CC…)
3. Train and compare 7 regression models
4. Evaluate with MAE, RMSE, R² and Cross-Validation
5. Deploy an interactive prediction interface


## 1. 📦 Imports & Setup

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline
print('✅ All imports successful')

## 2. 📋 Load Dataset
> Place `car_data.csv` (from Kaggle) in the `data/` folder, or run `python data/generate_dataset.py` to create a synthetic version.

In [ ]:
df = pd.read_csv('data/car_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

## 3. 📊 Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable — Selling Price', fontsize=14, fontweight='bold')
axes[0].hist(df['Selling_Price'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Original Distribution'); axes[0].set_xlabel('Price (₹ Lakhs)')
axes[1].hist(np.log1p(df['Selling_Price']), bins=30, color='teal', edgecolor='white')
axes[1].set_title('Log-Transformed'); axes[1].set_xlabel('log(1+Price)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=np.triu(np.ones_like(corr, dtype=bool)), ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('Fuel_Type')['Selling_Price'].median().sort_values(ascending=False)\
  .plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'), edgecolor='white')
axes[0].set_title('Median Price by Fuel Type')
df.groupby('Transmission')['Selling_Price'].median().sort_values(ascending=False)\
  .plot(kind='bar', ax=axes[1], color=sns.color_palette('Set3'), edgecolor='white')
axes[1].set_title('Median Price by Transmission')
plt.tight_layout(); plt.show()

## 4. 🔧 Feature Engineering

In [ ]:
# Derived features
df['Car_Age']       = 2020 - df['Year']
df['Price_Ratio']   = df['Selling_Price'] / df['Present_Price']
df['Kms_Per_Year']  = df['Kms_Driven'] / (df['Car_Age'] + 1)
df['Power_Per_CC']  = df['Max_Power'] / df['Engine']

goodwill_map = {
    'Maruti': 0.95, 'Hyundai': 0.90, 'Honda': 0.88, 'Toyota': 0.92,
    'Ford': 0.82, 'Volkswagen': 0.85, 'Skoda': 0.80, 'Audi': 0.75,
    'BMW': 0.72, 'Mercedes': 0.70, 'Mahindra': 0.78, 'Tata': 0.76,
    'Renault': 0.79, 'Datsun': 0.77, 'Chevrolet': 0.74, 'Fiat': 0.73,
    'Jaguar': 0.68, 'Land_Rover': 0.65, 'Jeep': 0.71, 'Nissan': 0.80,
}
df['Brand_Goodwill'] = df['Car_Name'].map(goodwill_map).fillna(0.75)

# Encode categoricals
le = LabelEncoder()
df['Fuel_Type_enc']    = le.fit_transform(df['Fuel_Type'])
df['Seller_Type_enc']  = le.fit_transform(df['Seller_Type'])
df['Transmission_enc'] = le.fit_transform(df['Transmission'])

print('✅ Features engineered. New shape:', df.shape)
df[['Car_Age','Price_Ratio','Kms_Per_Year','Power_Per_CC','Brand_Goodwill']].head()

## 5. 🤖 Model Training & Evaluation

In [ ]:
features = [
    'Present_Price','Kms_Driven','Car_Age','Owner',
    'Fuel_Type_enc','Seller_Type_enc','Transmission_enc',
    'Mileage','Engine','Max_Power','Seats',
    'Brand_Goodwill','Kms_Per_Year','Power_Per_CC',
]
X, y = df[features].fillna(df[features].median()), df['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(alpha=1.0),
    'Lasso'             : Lasso(alpha=0.1),
    'Decision Tree'     : DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
    'AdaBoost'          : AdaBoostRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, m in models.items():
    sc = name in ('Linear Regression','Ridge','Lasso')
    Xtr, Xte = (X_train_sc, X_test_sc) if sc else (X_train, X_test)
    m.fit(Xtr, y_train); p = m.predict(Xte)
    results[name] = {
        'MAE' : mean_absolute_error(y_test, p),
        'RMSE': np.sqrt(mean_squared_error(y_test, p)),
        'R²'  : r2_score(y_test, p),
        'CV R²': cross_val_score(m, Xtr, y_train, cv=5, scoring='r2').mean(),
        'preds': p
    }

res_df = pd.DataFrame({k:{m:v[m] for m in ('MAE','RMSE','R²','CV R²')} for k,v in results.items()}).T
res_df.round(4).style.highlight_max(axis=0, subset=['R²','CV R²'], color='#c6f0c2')\
                     .highlight_min(axis=0, subset=['MAE','RMSE'],  color='#c6f0c2')

## 6. 📈 Visualisation — Actual vs Predicted

In [ ]:
best = res_df['R²'].idxmax()
preds = results[best]['preds']
print(f'🏆 Best model: {best}  R²={res_df.loc[best,"R²"]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, preds, alpha=0.6, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect Fit')
axes[0].set_xlabel('Actual Price'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'{best} — Actual vs Predicted'); axes[0].legend()
residuals = y_test.values - preds
axes[1].hist(residuals, bins=25, color='coral', edgecolor='white')
axes[1].axvline(0, color='k', ls='--')
axes[1].set_title('Residual Distribution')
plt.tight_layout(); plt.show()

## 7. 🔍 Feature Importance

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
fi.plot(kind='barh', ax=ax, color=sns.color_palette('Blues_r', len(fi)))
ax.set_title('Feature Importances — Random Forest')
plt.tight_layout(); plt.show()

## 8. 🚀 Make a Prediction

In [ ]:
import joblib
best_model = models[best]
joblib.dump(best_model, 'models/best_model.pkl')
joblib.dump(scaler,     'models/scaler.pkl')

# Sample prediction
sample = pd.DataFrame([{
    'Present_Price':8.0,'Kms_Driven':45000,'Car_Age':4,'Owner':0,
    'Fuel_Type_enc':2,'Seller_Type_enc':0,'Transmission_enc':0,
    'Mileage':18.5,'Engine':1197,'Max_Power':82.0,'Seats':5,
    'Brand_Goodwill':0.90,'Kms_Per_Year':11250.0,'Power_Per_CC':0.069,
}])
pred = best_model.predict(sample[features])[0]
print(f'💰 Predicted Price for sample car: ₹ {pred:.2f} Lakhs')